# P1 — What is over-squashing?

A GNN updates each node by mixing in its neighbours, **one hop at a time**. To reach information that is `g`
hops away, you need `g` layers. The trouble: if many paths funnel into a node, a huge number of messages must
be packed into that node's single **fixed-size vector**. When there is more to pack than the vector can hold,
information is lost — that is **over-squashing**.

**5 figures:** the bottleneck graph · a message hopping layer-by-layer · the *overflowing vector* · the message
explosion · where the messages come from.

In [ ]:
import torch, numpy as np
from oversquash import viz
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 4, 3

## Figure 1 — the bottleneck

Sources (blue) feed through a narrow middle (yellow) into one target (green). **Every** path crosses the
narrow part — that is the bottleneck.

In [ ]:
data = make_bottleneck_graph(K, M, 2, torch.Generator().manual_seed(0))
viz.draw_bottleneck(data, K, M, title='sources -> bottleneck -> target')

## Figure 2 — a message travels one hop per layer

Information moves step by step. To get from a far source to the target takes as many layers as the path is
long. Watch the highlighted message advance one hop in each panel.

In [ ]:
dia = [(0,1),(0,2),(1,3),(2,3)]; pos = {0:(0,0),1:(1,1),2:(1,-1),3:(2,0)}
viz.explain_message_hops(dia, pos, path=[(0,1),(1,3)], n_nodes=4, src=0, dst=3,
                         title='one message, one hop per layer')

## Figure 3 — the overflowing vector (this *is* over-squashing)

The target's vector has a **fixed capacity**. As the graph gets deeper, the number of messages it must absorb
grows fast. Once they exceed the capacity, the vector overflows and information is lost. This single picture is
the whole problem.

In [ ]:
msgs = []
for d in [1,2,3,4]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0]); msgs.append(int(raw[d][:, t].sum()))
viz.explain_overflow([(d, m) for d, m in zip([1,2,3,4], msgs)], capacity=12,
                     title='fixed-capacity vector vs. growing messages')

## Figure 4 — how fast the messages explode

The same counts on a log scale: the number of messages grows like `K·M^depth` — exponentially with distance.

In [ ]:
viz.plot_message_explosion([1,2,3,4], msgs, 'messages squashed into one vector', 'depth', 'messages (log)')

## Figure 5 — where the messages come from

The multiplicity matrix at the deepest range: row = target, column = source, colour = number of walks. The
bright column into the target is the pile-up that gets squashed. **Next (P2): how to count these paths exactly.**

In [ ]:
deep = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, _ = walk_operators(deep.edge_index.numpy(), deep.num_nodes, max_length=4)
viz.plot_multiplicity_heatmap(raw[3], 'walk multiplicity at range 4 (A^4)')